## break out by top usage class/ material type - this will inform table schema

In [2]:
import glob
import pandas as pd

checkpoint_dir = '/Users/audriswong/data-portfolio/projects/seattle-checkouts/data/raw/checkpoints'

# get material types
files = glob.glob(f'{checkpoint_dir}/*.parquet')
df_all = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

pd.set_option('display.max_rows', 100)  # ensure all 67 rows print
df_all.groupby(['usageclass', 'materialtype']).size().reset_index(name='row_count').sort_values('row_count', ascending=False)

,usageclass,materialtype,row_count
13,Physical,BOOK,9329231
3,Digital,EBOOK,6874896
0,Digital,AUDIOBOOK,3376834
64,Physical,VIDEODISC,1594039
55,Physical,SOUNDDISC,1314824
8,Digital,SONG,257905
4,Digital,Ebook,210743
1,Digital,Audiobook,126160
44,Physical,REGPRINT,54175
7,Digital,MUSIC,48846


## Majority of checkouts are physical books, video disks, sound desks, and digital ebooks and audiobooks.  Break out into physical vs digital checkout tables, normalize case in materialtype

## Confirm it's safe to collapse materialtype column so that values are not case sensitive

In [2]:
# Normalize to uppercase and check if counts change
df_all['materialtype_upper'] = df_all['materialtype'].str.upper().str.strip()

# Compare original vs normalized counts
original = df_all['materialtype'].value_counts()
normalized = df_all['materialtype_upper'].value_counts()

print("Original:")
print(original)
print("\nNormalized to uppercase:")
print(normalized)

# Show which types get collapsed
print("\nTypes that get collapsed:")
for norm_type in normalized.index:
    originals = df_all[df_all['materialtype_upper'] == norm_type]['materialtype'].unique()
    if len(originals) > 1:
        print(f"  {norm_type}: {originals}")

Original:
materialtype
BOOK                                          9329231
EBOOK                                         6874896
AUDIOBOOK                                     3376834
VIDEODISC                                     1594039
SOUNDDISC                                     1314824
SONG                                           257905
Ebook                                          210743
Audiobook                                      126160
MUSIC                                           88264
REGPRINT                                        54175
MOVIE                                           37157
TELEVISION                                      28529
COMIC                                           22238
LARGEPRINT                                      13704
SOUNDDISC, VIDEODISC                            12840
VIDEO                                            9762
SOUNDREC                                         7642
CR                                               6009
ER, S

## Clean + Store data into physical_checkouts and digital_checkouts, preserving all existing columns

### new columns added: generated work_key (title + creator), month_date

### cleaning includes removing whitespaces and turning characters upcase
### specifically for physical book titles (which contain creator metadata), parse out the title only
#### see QA file for more details

In [3]:
import re

processed_dir = '/Users/audriswong/data-portfolio/projects/seattle-checkouts/data/processed'

# Normalize materialtype
df_all['materialtype'] = df_all['materialtype'].str.upper().str.strip()
print(f"Distinct material types after normalize: {df_all['materialtype'].nunique()}")

# Normalize title and creator before generating key
# title_clean: remove whitespace, uppercase
# creator_clean: remove whitespace, uppercase, remove birth/death years (YYYY- or YYYY-YYYY)
# title_book_clean: for PHYSICAL BOOK rows only — extract core title before ' / ' separator
#                   which typically separates title from author/illustrator credits

def extract_core_title(title):
    """For physical books: return first part of title before ' / ' separator.
    Leaves titles without ' / ' unchanged."""
    if not title or str(title) == 'None':
        return title
    core = str(title).split(' / ')[0].strip()
    core = re.sub(r'[\.\,]+$', '', core).strip()  # remove trailing punctuation
    return core.upper()

df_all['title_clean'] = df_all['title'].str.strip().str.upper()
df_all['creator_clean'] = (df_all['creator']
                            .str.strip()
                            .str.upper()
                            .apply(lambda x: re.sub(r',?\s*\d{4}-?(\d{4})?', '', str(x)).strip()))

# Apply book-specific title cleaning only to PHYSICAL BOOK rows
book_mask = (
    (df_all['usageclass'].str.upper().str.strip()   == 'PHYSICAL') &
    (df_all['materialtype'].str.upper().str.strip() == 'BOOK')
)
df_all['title_book_clean'] = df_all['title_clean']  # default: same as title_clean for all rows
df_all.loc[book_mask, 'title_book_clean'] = df_all.loc[book_mask, 'title'].apply(extract_core_title)

# Convert checkouts, checkoutmonth, and checkoutyear to numeric
df_all['checkouts']     = pd.to_numeric(df_all['checkouts'],     errors='coerce')
df_all['checkoutmonth'] = pd.to_numeric(df_all['checkoutmonth'], errors='coerce')
df_all['checkoutyear']  = pd.to_numeric(df_all['checkoutyear'],  errors='coerce')

# Generate composite key
df_all['work_key'] = df_all['title_clean'] + '||' + df_all['creator_clean']

# Generate month_date
df_all['month_date'] = pd.to_datetime(
    df_all['checkoutyear'].astype(str) + '-' +
    df_all['checkoutmonth'].astype(str).str.zfill(2) + '-01'
)

# Check for nulls after numeric conversion
print(f"\nRows with non-numeric checkouts: {df_all['checkouts'].isna().sum():,}")
print(f"Rows with non-numeric month:     {df_all['checkoutmonth'].isna().sum():,}")
print(f"Rows with non-numeric year:      {df_all['checkoutyear'].isna().sum():,}")

# Split and save
for usage_class, group_df in df_all.groupby('usageclass'):
    table_name  = f"{usage_class.lower().strip()}_checkouts"
    output_path = f"{processed_dir}/{table_name}.parquet"
    group_df.reset_index(drop=True).to_parquet(output_path)
    print(f"\n✅ {table_name}")
    print(f"   Rows:           {len(group_df):,}")
    print(f"   Material types: {sorted(group_df['materialtype'].unique())}")

print("\n🎉 Done!")

Distinct material types after normalize: 64

Rows with non-numeric checkouts: 0
Rows with non-numeric month:     0
Rows with non-numeric year:      0

✅ digital_checkouts
   Rows:           10,996,363
   Material types: ['AUDIOBOOK', 'COMIC', 'EBOOK', 'MAGAZINE', 'MOVIE', 'MUSIC', 'SONG', 'TELEVISION', 'VIDEO']

✅ physical_checkouts
   Rows:           12,397,324
   Material types: ['ATLAS', 'ATLAS, ER', 'BOOK', 'BOOK, ER', 'CHART', 'COMPFILE', 'CR', 'ER', 'ER, MAP', 'ER, NONPROJGRAPH', 'ER, PRINT', 'ER, REGPRINT', 'ER, SOUNDDISC', 'ER, SOUNDDISC, SOUNDREC', 'ER, SOUNDREC', 'ER, VIDEODISC', 'ER, VIDEOREC', 'FLASHCARD', 'FLASHCARD, SOUNDDISC', 'GLOBE', 'KIT', 'LARGEPRINT', 'MAP', 'MAP, VIEW', 'MICROFORM', 'MIXED', 'MUSIC', 'MUSICSNDREC', 'NONPROJGRAPH', 'NOTATEDMUSIC', 'PHOTO', 'PICTURE', 'PRINT', 'REGPRINT', 'REGPRINT, SOUNDDISC', 'REGPRINT, VIDEOREC', 'REMOTESEN', 'SLIDE', 'SLIDE, SOUNDCASS', 'SLIDE, SOUNDCASS, VIDEOCASS', 'SOUNDCASS', 'SOUNDCASS, SOUNDDISC', 'SOUNDCASS, SOUNDDISC, SOU

# Create Book and EBook Checkout tables

In [15]:
#import, setup, clean
processed_dir = '/Users/audriswong/data-portfolio/projects/seattle-checkouts/data/processed'

#import data
physical_checkouts = pd.read_parquet(f'{processed_dir}/physical_checkouts.parquet')
digital_checkouts = pd.read_parquet(f'{processed_dir}/digital_checkouts.parquet')


#import data 
#pulls in the book specific cleaned title as title_clean, recreates work_key using the right cleaned title

book_checkouts = (physical_checkouts[physical_checkouts['materialtype'] == 'BOOK']
                  .drop(columns=['title_clean', 'work_key'])
                  .rename(columns={'title_book_clean': 'title_clean',
                                   'title': 'title_raw'}))

ebook_checkouts = (digital_checkouts[digital_checkouts['materialtype'] == 'EBOOK']
                   .drop(columns=['work_key', 'title_book_clean'])
                   .rename(columns={'title': 'title_raw'}))

book_checkouts['work_key'] = book_checkouts['title_clean'] + ' || ' + book_checkouts['creator_clean']
ebook_checkouts['work_key'] = ebook_checkouts['title_clean'] + ' || ' + ebook_checkouts['creator_clean']

ebook_checkouts['title_clean'] = ebook_checkouts['title_clean'].str.upper()

In [16]:
# extract genre of the book based on subjects

#extract genre from keywords
# See a sample of raw subjects values
print("Sample subjects values:")
book_checkouts['subjects'].dropna().sample(20).tolist()

# Check null rate
null_rate = book_checkouts['subjects'].isna().mean() * 100
print(f"Null subjects: {null_rate:.1f}%")

# See most common subject strings
print("\nMost common subjects:")
book_checkouts['subjects'].value_counts().head(20)

#keyword -> genre mapping
genre_keywords = {
    'Mystery/Thriller': ['mystery', 'detective', 'crime', 'thriller', 'suspense'],
    'Science Fiction':  ['science fiction', 'sci-fi'],
    'Fantasy':          ['fantasy'],
    'Romance':          ['romance', 'love stories'],
    'Horror':           ['horror'],
    'Historical':       ['historical fiction', 'history'],
    'Biography/Memoir': ['biography', 'memoir', 'autobiography'],
    'Self-Help':        ['self-help', 'personal development', 'motivation', 'success'],
    'Cooking':          ['cooking', 'cookbooks', 'baking', 'vegan cooking', 'food'],
    'Poetry':           ['poetry'],
    'Graphic Novel':    ['graphic novels', 'comics', 'manga'],
    'Juvenile':         ['juvenile', 'childrens', "children's", 'picture book', 'young adult'],
    'Crafts':           ['knitting', 'quilting', 'sewing', 'crafts'],
    'Health/Wellness':  ['health', 'wellness', 'fitness', 'diet', 'medicine'],
    'Business':         ['business', 'economics', 'finance', 'management'],
    'Science/Nature':   ['science', 'nature', 'animals', 'dinosaurs'],
    'Religion':         ['religion', 'spiritual', 'christian', 'bible'],
    'Travel':           ['travel', 'geography'],
    'Art/Design':       ['art', 'interior decoration', 'design', 'photography'],
    'Humor':            ['humor'],
    'Nonfiction':       ['nonfiction'],
    'Fiction':          ['fiction', 'novel', 'short stories', 'literature'],
}

def extract_genre(subjects):
    if pd.isna(subjects):
        return 'Unknown'
    subjects_lower = str(subjects).lower()
    for genre, keywords in genre_keywords.items():
        if any(kw in subjects_lower for kw in keywords):
            return genre
    return 'Other'

book_checkouts['genre'] = book_checkouts['subjects'].apply(extract_genre)
ebook_checkouts['genre'] = ebook_checkouts['subjects'].apply(extract_genre)

# Check coverage
print("Physical Books - Genre distribution:")
print(book_checkouts['genre'].value_counts())

print(f"\nUnclassified (Other/Unknown): {(book_checkouts['genre'].isin(['Other','Unknown'])).mean()*100:.1f}%")

# Spot check 'Other' to see what's being missed
print("\nPhysical Books - Sample unclassified subjects:")
book_checkouts[book_checkouts['genre'] == 'Other']['subjects'].value_counts().head(20)

print("Digital EBooks - Genre distribution:")
print(ebook_checkouts['genre'].value_counts())

print(f"\nUnclassified (Other/Unknown): {(ebook_checkouts['genre'].isin(['Other','Unknown'])).mean()*100:.1f}%")

# Spot check 'Other' to see what's being missed
print("\nDigital EBooks - Sample unclassified subjects:")
ebook_checkouts[ebook_checkouts['genre'] == 'Other']['subjects'].value_counts().head(20)

Sample subjects values:
Null subjects: 1.5%

Most common subjects:
Physical Books - Genre distribution:
genre
Juvenile            2561198
Other               1379402
Mystery/Thriller     834843
Historical           802059
Fiction              459614
Biography/Memoir     459455
Fantasy              386871
Cooking              384427
Graphic Novel        380479
Science Fiction      273977
Art/Design           236081
Romance              215858
Unknown              138366
Poetry               110773
Business             110342
Travel               103777
Health/Wellness       94219
Religion              86051
Science/Nature        80169
Horror                63110
Humor                 57999
Crafts                56152
Self-Help             52039
Nonfiction             1970
Name: count, dtype: int64

Unclassified (Other/Unknown): 16.3%

Physical Books - Sample unclassified subjects:
Digital EBooks - Genre distribution:
genre
Mystery/Thriller    1489932
Juvenile             912543
Romance 

subjects
Beginning Reader                   489
Comic and Graphic Books            166
Essays                              21
Mathematics                         10
Family & Relationships               8
Children                             7
NA                                   2
Self-Improvement                     2
Military                             2
New Age                              1
Sociology, Sports & Recreations      1
Biology                              1
Name: count, dtype: int64

In [17]:
print(f"=== Book Checkouts sample===")
print(book_checkouts.head())
print(f"\n=== EBook Checkouts sample===")
print(ebook_checkouts.head())

=== Book Checkouts sample===
  usageclass checkouttype materialtype  checkoutyear  checkoutmonth  \
0   Physical      Horizon         BOOK          2019              7   
2   Physical      Horizon         BOOK          2019              7   
3   Physical      Horizon         BOOK          2019              7   
4   Physical      Horizon         BOOK          2019              7   
5   Physical      Horizon         BOOK          2019              7   

   checkouts                                          title_raw  \
0          2  Microbiology / designed and created by Basher ...   
2          2  People who eat darkness : the true story of a ...   
3          2                           Tea Rex / by Molly Idle.   
4          1  Portuguese tiles from the National Museum of A...   
5          1  Wan an, hui jia luo! / wen/tu, Gongye Xiaozi ;...   

                       creator  \
0    Green, Dan, 1975 June 20-   
2         Lloyd Parry, Richard   
3           Idle, Molly Schaar   
4  Pe

In [18]:
#save as parquets

book_checkouts.reset_index(drop=True).to_parquet(f"{processed_dir}/book_checkouts.parquet")
ebook_checkouts.reset_index(drop=True).to_parquet(f"{processed_dir}/ebook_checkouts.parquet")
print("✅ book_checkouts and ebook_checkouts saved")



✅ book_checkouts and ebook_checkouts saved
